In [ ]:
# 1. 라이브러리 및 데이터 로드 (Phase 26: 950점 돌파를 위한 극한의 최적화)
import pandas as pd
import numpy as np
import os
import joblib
import json
import shutil
import warnings
warnings.filterwarnings('ignore')

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

DATA_DIR = r"C:\Users\이호준\OneDrive\바탕 화면\LG aimers\open\data"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "./data"

ID_COL = "row_id"
TARGET_COL = "control_success"

# 🔥 신규: 투수팀, 타자팀 ID를 범주형으로 추가 (팀 단위 고유의 전술/불펜 특성 학습)
CAT_COLS = ["top_bottom", "game_type", "base_state", "pitcher_hand", "batter_hand", "count_state", "pitcher_team_id", "batter_team_id"]

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), encoding="utf-8-sig")
test_cols = pd.read_csv(os.path.join(DATA_DIR, "test.csv"), encoding="utf-8-sig", nrows=0).columns

# 기존 피처
train['count_state'] = train['balls_before'].astype(str) + "-" + train['strikes_before'].astype(str)
train['platoon_advantage'] = (train['pitcher_hand'] == train['batter_hand']).astype(int)
train['score_diff_abs'] = train['score_diff_pitcher_team'].abs()

# 순수 세이버메트릭스 지표 (결측치 없는 파워 피처)
train['count_adv'] = train['strikes_before'] - train['balls_before']
train['pitcher_cmd'] = train['asof_pitcher_strike_rate'] / (train['asof_pitcher_ball_rate'] + 0.001)
train['pitcher_fastball_stuff'] = train['asof_pitcher_fastball_rate'] * train['asof_pitcher_success_rate']

# 글로벌 트랙맨 (노이즈 방지)
tm_stats = pd.read_csv(os.path.join(DATA_DIR, "pitcher_trackman_stats_v2.csv"))
train = pd.merge(train, tm_stats, on='pitcher_id', how='left')
tm_features = ['tm_rel_speed', 'tm_spin_rate', 'tm_ivb', 'tm_hb', 'tm_extension', 
               'tm_rel_height', 'tm_rel_side', 'tm_rel_speed_std', 'tm_hb_std', 'tm_ivb_std']

FEATURES = [c for c in test_cols if c != ID_COL] + ['count_state', 'platoon_advantage', 'score_diff_abs', 'count_adv', 'pitcher_cmd', 'pitcher_fastball_stuff'] + tm_features
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

print(f"✅ 팀 ID 범주화 및 피처 추가 완료! (총 피처 수: {len(FEATURES)})")


In [ ]:
# 2. 전처리 파이프라인
preprocessor = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT_COLS),
    ("num", SimpleImputer(strategy="median"), NUM_COLS),
])


In [ ]:
# 3. 2024년 퓨어 Isotonic 보정 및 Mega Ensemble (n_estimators=1500, seeds=10)
is_val = train["season"] == 2024
X_train, y_train = train.loc[~is_val, FEATURES], train.loc[~is_val, TARGET_COL].values
X_val, y_val = train.loc[is_val, FEATURES], train.loc[is_val, TARGET_COL].values

print("전처리 적용 중...")
preprocessor.fit(X_train)
X_train_pre = preprocessor.transform(X_train)
X_val_pre = preprocessor.transform(X_val)

# 🔥 시드 10개로 분산 완벽 제거 (Mega Ensemble)
SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
ensemble_preds = np.zeros(len(X_val))

print(f"🔥 {len(SEEDS)*3}개 메가 앙상블 학습 중... (학습률 0.01, 트리 1500개)")
for s in SEEDS:
    # 파라미터 극한 강화
    lgb_m = lgb.LGBMClassifier(n_estimators=1500, learning_rate=0.01, num_leaves=63, max_depth=8, min_child_samples=500, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=s, verbosity=-1)
    lgb_m.fit(X_train_pre, y_train)
    p_lgb = lgb_m.predict_proba(X_val_pre)[:, 1]
    
    xgb_m = xgb.XGBClassifier(n_estimators=1500, learning_rate=0.01, max_depth=8, min_child_weight=500, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=s, tree_method='hist')
    xgb_m.fit(X_train_pre, y_train)
    p_xgb = xgb_m.predict_proba(X_val_pre)[:, 1]
    
    cb_m = cb.CatBoostClassifier(iterations=1500, learning_rate=0.01, depth=8, min_data_in_leaf=500, random_seed=s, verbose=0)
    cb_m.fit(X_train_pre, y_train)
    p_cb = cb_m.predict_proba(X_val_pre)[:, 1]
    
    ensemble_preds += (p_lgb + p_xgb + p_cb) / 3.0

ensemble_preds /= len(SEEDS)

print("\n🔧 2024년 정통 Isotonic 보정기 가동 중...")
iso = IsotonicRegression(out_of_bounds='clip')
calibrated_preds = iso.fit_transform(ensemble_preds, y_val)

r = y_val.mean()
brier = ((calibrated_preds - y_val) ** 2).mean()
baseline_brier = r * (1 - r)
score = max(0, 100000 * (1 - brier / baseline_brier))

print(f"\n✅ Brier Score: {brier:.6f} | 기준선 r(1-r): {baseline_brier:.6f}")
print(f"🚀 Sabermetrics Validation Score: {score:.2f}")

iso_data = {
    "X_thresh": iso.X_thresholds_.tolist(),
    "y_thresh": iso.y_thresholds_.tolist()
}
os.makedirs("./submission/model", exist_ok=True)
with open("./submission/model/iso_thresholds.json", "w") as f:
    json.dump(iso_data, f)


전처리 적용 중...
🔥 30개 메가 앙상블 학습 중... (학습률 0.01, 트리 1500개)


KeyboardInterrupt: 

811.49점

In [ ]:
# 4. 베이스 모델 최종 진화(100% 데이터) 및 스크립트 생성
import zipfile

print("전체 데이터 전처리 및 메가 앙상블 최종 학습 중... (매우 오래 걸립니다)")
preprocessor.fit(train[FEATURES])
joblib.dump(preprocessor, "./submission/model/preprocessor.pkl", compress=3)
joblib.dump(FEATURES, "./submission/model/features.pkl")
shutil.copy(os.path.join(DATA_DIR, "pitcher_trackman_stats_v2.csv"), "./submission/model/pitcher_trackman_stats_global.csv")

X_all_pre = preprocessor.transform(train[FEATURES])
y_all = train[TARGET_COL].values

# 🔥 신규: 2024년(ABS 룰) 데이터에 가중치 3배 부여 (Domain Adaptation)
# 2025년을 예측해야 하므로, 동일하게 ABS가 적용되었던 2024년 데이터의 중요도를 강제로 끌어올립니다.
weights_all = np.where(train['season'] == 2024, 3.0, 1.0)

for i, s in enumerate(SEEDS):
    lgb_m = lgb.LGBMClassifier(n_estimators=1500, learning_rate=0.01, num_leaves=63, max_depth=8, min_child_samples=500, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=s, verbosity=-1)
    lgb_m.fit(X_all_pre, y_all, sample_weight=weights_all)
    lgb_m.booster_.save_model(f"./submission/model/lgb_seed{i}.txt")
    
    xgb_m = xgb.XGBClassifier(n_estimators=1500, learning_rate=0.01, max_depth=8, min_child_weight=500, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=s, tree_method='hist')
    xgb_m.fit(X_all_pre, y_all, sample_weight=weights_all)
    xgb_m.save_model(f"./submission/model/xgb_seed{i}.json")
    
    cb_m = cb.CatBoostClassifier(iterations=1500, learning_rate=0.01, depth=8, min_data_in_leaf=500, random_seed=s, verbose=0)
    cb_m.fit(X_all_pre, y_all, sample_weight=weights_all)
    cb_m.save_model(f"./submission/model/cb_seed{i}.cbm")

print(f"{len(SEEDS)*3}개 모델 학습 및 저장 완료!")

with open("./submission/requirements.txt", "w") as f:
    f.write("pandas\nscikit-learn\nlightgbm\nxgboost\ncatboost\njoblib\n")

script_code = """import os
import joblib
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

ID_COL = "row_id"
TARGET_COL = "control_success"
N_SEEDS = 10

def merge_predictions(sub, ids, preds):
    pred_map = dict(zip(ids, preds))
    values = [pred_map.get(rid, cur) for rid, cur in zip(sub[ID_COL], sub[TARGET_COL])]
    sub[TARGET_COL] = values
    return sub

def main():
    TEST_PATH = "./data/test.csv"
    SAMPLE_SUB_PATH = "./data/sample_submission.csv"
    OUT_PATH = "./output/submission.csv"

    preprocessor = joblib.load("./model/preprocessor.pkl")
    FEATURES = joblib.load("./model/features.pkl")
    tm_stats = pd.read_csv("./model/pitcher_trackman_stats_global.csv")
    
    with open("./model/iso_thresholds.json", "r") as f:
        iso_data = json.load(f)
    X_thresh = np.array(iso_data["X_thresh"])
    y_thresh = np.array(iso_data["y_thresh"])
    
    test = pd.read_csv(TEST_PATH, encoding="utf-8-sig")
    sub = pd.read_csv(SAMPLE_SUB_PATH, encoding="utf-8-sig")
    
    test['count_state'] = test['balls_before'].astype(str) + "-" + test['strikes_before'].astype(str)
    test['platoon_advantage'] = (test['pitcher_hand'] == test['batter_hand']).astype(int)
    test['score_diff_abs'] = test['score_diff_pitcher_team'].abs()
    
    test['count_adv'] = test['strikes_before'] - test['balls_before']
    test['pitcher_cmd'] = test['asof_pitcher_strike_rate'] / (test['asof_pitcher_ball_rate'] + 0.001)
    test['pitcher_fastball_stuff'] = test['asof_pitcher_fastball_rate'] * test['asof_pitcher_success_rate']
    
    test = pd.merge(test, tm_stats, on='pitcher_id', how='left')
    
    ids = test[ID_COL].tolist()
    X_pre = preprocessor.transform(test[FEATURES])
    
    final_preds = np.zeros(len(test))
    
    for i in range(N_SEEDS):
        lgb_booster = lgb.Booster(model_file=f"./model/lgb_seed{i}.txt")
        xgb_model = xgb.XGBClassifier()
        xgb_model.load_model(f"./model/xgb_seed{i}.json")
        cb_model = cb.CatBoostClassifier()
        cb_model.load_model(f"./model/cb_seed{i}.cbm")
        
        p_lgb = lgb_booster.predict(X_pre)
        p_xgb = xgb_model.predict_proba(X_pre)[:, 1]
        p_cb = cb_model.predict_proba(X_pre)[:, 1]
        
        final_preds += (p_lgb + p_xgb + p_cb) / 3.0
        
    final_preds /= N_SEEDS
    
    calibrated_preds = np.interp(final_preds, X_thresh, y_thresh)
    
    sub = merge_predictions(sub, ids, calibrated_preds)
    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    sub.to_csv(OUT_PATH, index=False, encoding="utf-8")

if __name__ == "__main__":
    main()
"""

with open("./submission/script.py", "w", encoding="utf-8") as f:
    f.write(script_code)

import zipfile
zip_path = 'phase26_abs_domain_mega_950.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk('./submission'):
        for file in files:
            file_path = os.path.join(root, file)
            zipf.write(file_path, os.path.relpath(file_path, './submission'))

print("✅ 도메인 가중치 및 메가 앙상블 적용! phase26_abs_domain_mega_950.zip 생성 완료!")


전체 데이터 전처리 및 메가 앙상블 최종 학습 중... (매우 오래 걸립니다)


NameError: name 'SEEDS' is not defined